In [ ]:
import sys
import os
import time
import yaml
sys.path.append(os.path.abspath('..')) 
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pybullet as p

STAGE_TO_WATCH = 0

print(f"Best Model Tracker initialized. Monitoring Stage {STAGE_TO_WATCH}...")

config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]
reward_weights = config['rewards']

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
RUN_NAME = stage_config['run_name']

env_raw = RoomDroneEnv(gui=True, num_obstacles=NUM_OBS, randomize_obstacles=RAND_OBS, randomize_coins=RAND_COINS, reward_weights=reward_weights)
env_vec = DummyVecEnv([lambda e=env_raw: e])

model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'best_model'))
vecnorm_path = f"{model_path}_vecnormalize.pkl"
print(f"Waiting for best brain at: {model_path}.zip")

while not (os.path.exists(f"{model_path}.zip") and os.path.exists(vecnorm_path)):
    print("Waiting for the first EvalCallback to create best_model.zip and .pkl...")
    time.sleep(5)

env = VecNormalize.load(vecnorm_path, env_vec)
env.training = False
env.norm_reward = False
model = PPO.load(model_path, env=env)

obs = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print(f"[{RUN_NAME}] Best Model Simulation started. Updating dynamically when records are broken...")

while True:
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, infos = env.step(action) 
    time.sleep(1./240.) 
    
    if dones[0]:
        time.sleep(0.5)
        try:
            # FIX: Reload VecNormalize stats alongside the model.
            # Previously only the model was reloaded; stale normalization stats
            # cause the watcher to feed incorrectly scaled observations after
            # training has updated the running mean/variance significantly.
            env = VecNormalize.load(vecnorm_path, env_vec)
            env.training = False
            env.norm_reward = False
            model = PPO.load(model_path, env=env)
            obs = env.reset()
        except Exception:
            pass